In [3]:
# Task 1: Load raw full 20 newsgroups dataset (code adapted from tutorial - https://scikit-learn.org/stable/auto_examples/text/plot_document_classification_20newsgroups.html)

from time import time
from sklearn.datasets import fetch_20newsgroups
import numpy as np  # Optional: for shapes

def size_mb(docs):
    return sum(len(s.encode("utf-8")) for s in docs) / 1e6

def load_raw_data(verbose=False, remove=()):
    """Load the 20 newsgroups dataset (raw, no vectorization)."""
    data_train = fetch_20newsgroups(
        subset="train",
        shuffle=True,
        random_state=42,
        remove=remove,
    )

    data_test = fetch_20newsgroups(
        subset="test",
        shuffle=True,
        random_state=42,
        remove=remove,
    )

    # order of labels in `target_names` can be different from `categories`
    target_names = data_train.target_names

    # split target in a training set and a test set
    y_train, y_test = data_train.target, data_test.target

    if verbose:
        data_train_size_mb = size_mb(data_train.data)
        data_test_size_mb = size_mb(data_test.data)
        print(f"{len(data_train.data)} documents - {data_train_size_mb:.2f}MB (training set)")
        print(f"{len(data_test.data)} documents - {data_test_size_mb:.2f}MB (test set)")
        print("Loaded Categories:")
        for i, cat in enumerate(target_names):
            print(f"{i}: {cat}")
        print(f"y_train shape: {np.unique(y_train, return_counts=True)}")  
        print(f"y_test shape: {np.unique(y_test, return_counts=True)}")

    return data_train, data_test, y_train, y_test, target_names

# Run for Task 1 (verbose to confirm)
data_train, data_test, y_train, y_test, target_names = load_raw_data(verbose=True)

11314 documents - 22.05MB (training set)
7532 documents - 13.80MB (test set)
Loaded Categories:
0: alt.atheism
1: comp.graphics
2: comp.os.ms-windows.misc
3: comp.sys.ibm.pc.hardware
4: comp.sys.mac.hardware
5: comp.windows.x
6: misc.forsale
7: rec.autos
8: rec.motorcycles
9: rec.sport.baseball
10: rec.sport.hockey
11: sci.crypt
12: sci.electronics
13: sci.med
14: sci.space
15: soc.religion.christian
16: talk.politics.guns
17: talk.politics.mideast
18: talk.politics.misc
19: talk.religion.misc
y_train shape: (array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19]), array([480, 584, 591, 590, 578, 593, 585, 594, 598, 597, 600, 595, 591,
       594, 593, 599, 546, 564, 465, 377]))
y_test shape: (array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19]), array([319, 389, 394, 392, 385, 395, 390, 396, 398, 397, 399, 396, 393,
       396, 394, 398, 364, 376, 310, 251]))


In [4]:
# Task 2: Comparison of three classifiers - ComplementNB, LogisticRegression, kNN 

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report  # To calculate Precision, Recall and F1
import pandas as pd

# Define the 3 classifiers
classifiers = [
    (ComplementNB(alpha=0.1), "Complement naive Bayes"),
    (LogisticRegression(C=5, max_iter=1000), "Logistic Regression"),
    (KNeighborsClassifier(n_neighbors=100), "kNN")
]

def benchmark(clf, X_train, X_test, y_train, y_test, name, target_names):
    
    t0 = time()
    clf.fit(X_train, y_train)
    duration = time() - t0
    y_pred = clf.predict(X_test)

    report = classification_report(y_test, y_pred, target_names=target_names, output_dict=True)
    macro_avg = report['macro avg']

    macro = {
        'precision_macro': macro_avg['precision'],
        'recall_macro': macro_avg['recall'],
        'f1_macro': macro_avg['f1-score']
    }
    macro['fit_time'] = duration
    print(f"{name}: F1-macro = {macro['f1_macro']:.3f} (fit in {duration:.2f}s)")
    return {name: macro}

# Vectorize with tutorial defaults (baseline)
vectorizer = TfidfVectorizer(
    sublinear_tf=True, max_df=0.5, min_df=5, stop_words="english"
)

X_train = vectorizer.fit_transform(data_train.data)
X_test = vectorizer.transform(data_test.data)
print(f"Baseline vectorization: {X_train.shape[1]:,} features")

# Run benchmarks and collect results
results = {}
for clf, name in classifiers:
    results.update(benchmark(clf, X_train, X_test, y_train, y_test, name, target_names))

df_task2 = pd.DataFrame([
    {'Classifier': name, 'Precision': r['precision_macro'], 'Recall': r['recall_macro'], 'F1': r['f1_macro']}
    for name, r in results.items()
]).round(3)
display(df_task2)

Baseline vectorization: 25,631 features
Complement naive Bayes: F1-macro = 0.819 (fit in 0.03s)
Logistic Regression: F1-macro = 0.842 (fit in 4.73s)
kNN: F1-macro = 0.753 (fit in 0.00s)


,Classifier,Precision,Recall,F1
0,Complement naive Bayes,0.829,0.820,0.819
1,Logistic Regression,0.847,0.841,0.842
2,kNN,0.772,0.755,0.753


In [5]:
# Task 3: Compare three types of features - counts, tf, and tf-idf

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Define vectorizers with common parameters
common_params = {
    'max_df': 0.5,
    'min_df': 5,
    'stop_words': "english"
}

vectorizers = [
    (CountVectorizer(**common_params), "Counts (CountVectorizer)"),
    (TfidfVectorizer(use_idf=False, sublinear_tf=True, **common_params), "TF (TfidfVectorizer without IDF)"),
    (TfidfVectorizer(sublinear_tf=True, **common_params), "TF-IDF (TfidfVectorizer)")
]

# Function to run benchmarks for a given vectorizer
def run_for_vectorizer(vectorizer, name, classifiers, data_train, data_test, y_train, y_test, target_names):
    print(f"\nVectorization with {name}:")
    X_train = vectorizer.fit_transform(data_train.data)
    X_test = vectorizer.transform(data_test.data)
    print(f"Vectorization: {X_train.shape[1]:,} features")
    
    results = {}
    for clf, clf_name in classifiers:
        results.update(benchmark(clf, X_train, X_test, y_train, y_test, clf_name, target_names))
    
    df = pd.DataFrame([
        {'Classifier': clf_name, 'Precision': r['precision_macro'], 'Recall': r['recall_macro'], 'F1': r['f1_macro']}
        for clf_name, r in results.items()
    ]).round(3)
    display(df)
    return results, df

# Run for each vectorizer
all_results_task3 = {}
all_dfs_task3 = []
for vec, vec_name in vectorizers:
    results, df = run_for_vectorizer(vec, vec_name, classifiers, data_train, data_test, y_train, y_test, target_names)
    all_results_task3[vec_name] = results
    df['Feature Type'] = vec_name  
    all_dfs_task3.append(df)

# Combined table for Task 3
df_task3_combined = pd.concat(all_dfs_task3).reset_index(drop=True)
display(df_task3_combined)

# Find the best combination (highest F1-macro)
best_f1 = 0
best_combo = None
for vec_name, res in all_results_task3.items():
    for clf_name, metrics in res.items():
        if metrics['f1_macro'] > best_f1:
            best_f1 = metrics['f1_macro']
            best_combo = (clf_name, vec_name)

print(f"\nBest combination: {best_combo[0]} with {best_combo[1]} (F1-macro = {best_f1:.3f})")


Vectorization with Counts (CountVectorizer):
Vectorization: 25,631 features
Complement naive Bayes: F1-macro = 0.795 (fit in 0.04s)
Logistic Regression: F1-macro = 0.793 (fit in 4.81s)
kNN: F1-macro = 0.313 (fit in 0.00s)


,Classifier,Precision,Recall,F1
0,Complement naive Bayes,0.813,0.800,0.795
1,Logistic Regression,0.798,0.792,0.793
2,kNN,0.613,0.316,0.313



Vectorization with TF (TfidfVectorizer without IDF):
Vectorization: 25,631 features
Complement naive Bayes: F1-macro = 0.828 (fit in 0.03s)
Logistic Regression: F1-macro = 0.814 (fit in 7.43s)
kNN: F1-macro = 0.694 (fit in 0.01s)


,Classifier,Precision,Recall,F1
0,Complement naive Bayes,0.841,0.829,0.828
1,Logistic Regression,0.819,0.814,0.814
2,kNN,0.718,0.697,0.694



Vectorization with TF-IDF (TfidfVectorizer):
Vectorization: 25,631 features
Complement naive Bayes: F1-macro = 0.819 (fit in 0.03s)
Logistic Regression: F1-macro = 0.842 (fit in 4.55s)
kNN: F1-macro = 0.753 (fit in 0.00s)


,Classifier,Precision,Recall,F1
0,Complement naive Bayes,0.829,0.820,0.819
1,Logistic Regression,0.847,0.841,0.842
2,kNN,0.772,0.755,0.753


,Classifier,Precision,Recall,F1,Feature Type
0,Complement naive Bayes,0.813,0.800,0.795,Counts (CountVectorizer)
1,Logistic Regression,0.798,0.792,0.793,Counts (CountVectorizer)
2,kNN,0.613,0.316,0.313,Counts (CountVectorizer)
3,Complement naive Bayes,0.841,0.829,0.828,TF (TfidfVectorizer without IDF)
4,Logistic Regression,0.819,0.814,0.814,TF (TfidfVectorizer without IDF)
5,kNN,0.718,0.697,0.694,TF (TfidfVectorizer without IDF)
6,Complement naive Bayes,0.829,0.820,0.819,TF-IDF (TfidfVectorizer)
7,Logistic Regression,0.847,0.841,0.842,TF-IDF (TfidfVectorizer)
8,kNN,0.772,0.755,0.753,TF-IDF (TfidfVectorizer)



Best combination: Logistic Regression with TF-IDF (TfidfVectorizer) (F1-macro = 0.842)


In [10]:
# Task 4: Experiment with parameters of the best classifier and feature from Task 3

from sklearn.linear_model import LogisticRegression  
import pandas as pd  

# Fixed best combo - Logistic Regression with TF-IDF (TfidfVectorizer)
best_clf = LogisticRegression(C=5, max_iter=1000)
best_vectorizer_class = TfidfVectorizer  
base_params = {'sublinear_tf': True, 'max_df': 0.5, 'min_df': 5, 'stop_words': 'english'}

# Function to run experiment
def experiment_with_params(params, exp_name, best_clf, data_train, data_test, y_train, y_test, target_names):
    vectorizer = best_vectorizer_class(**params)  
    X_train = vectorizer.fit_transform(data_train.data)  
    X_test = vectorizer.transform(data_test.data)  
    print(f"\nExperiment: {exp_name} - Features: {X_train.shape[1]:,}")
    
    
    res = benchmark(best_clf, X_train, X_test, y_train, y_test, "Logistic Regression", target_names)
    macro = res["Logistic Regression"]  
    
    return {
        'Experiment': exp_name,
        'Precision': macro['precision_macro'],
        'Recall': macro['recall_macro'],
        'F1': macro['f1_macro'],
        'Features': X_train.shape[1]  
    }

# Run experiments
experiments = []

# a. Lowercasing (true or false) - Default: True
for lowercase in [True, False]:
    params = {**base_params, 'lowercase': lowercase}
    exp_name = f"Lowercase: {lowercase}"
    experiments.append(experiment_with_params(params, exp_name, best_clf, data_train, data_test, y_train, y_test, target_names))

# b. Stop words (with or without) - Default: 'english'
for stop_words in ['english', None]:
    params = {**base_params, 'stop_words': stop_words}
    exp_name = f"Stop words: {'With' if stop_words else 'Without'}"
    experiments.append(experiment_with_params(params, exp_name, best_clf, data_train, data_test, y_train, y_test, target_names))

# c. Analyzer + ngram_range - Default: ('word', (1,1))
analyzer_ngrams = [
    ('word', (1,1)),     # Default unigrams
    ('word', (1,2)),     # Uni + bigrams
    ('word', (2,2)),     # Bigrams only
    ('char', (3,5))      # Char ngrams 3-5
]
for analyzer, ngram_range in analyzer_ngrams:
    params = {**base_params, 'analyzer': analyzer, 'ngram_range': ngram_range}
    exp_name = f"Analyzer: {analyzer}, ngram_range: {ngram_range}"
    experiments.append(experiment_with_params(params, exp_name, best_clf, data_train, data_test, y_train, y_test, target_names))

# d. max_features - Default: None (All)
max_features_list = [5000, 10000, 20000, None]
for max_feat in max_features_list:
    params = {**base_params, 'max_features': max_feat}
    exp_name = f"Max features: {max_feat if max_feat else 'All'}"
    experiments.append(experiment_with_params(params, exp_name, best_clf, data_train, data_test, y_train, y_test, target_names))

# Results table for Task 4
df_task4 = pd.DataFrame(experiments).round(3)
print("\nTask 4: Parameter Experiments Table (on Logistic Regression + TF-IDF)")
display(df_task4)  


Experiment: Lowercase: True - Features: 25,631
Logistic Regression: F1-macro = 0.842 (fit in 4.72s)

Experiment: Lowercase: False - Features: 30,935
Logistic Regression: F1-macro = 0.840 (fit in 9.70s)

Experiment: Stop words: With - Features: 25,631
Logistic Regression: F1-macro = 0.842 (fit in 5.15s)

Experiment: Stop words: Without - Features: 25,914
Logistic Regression: F1-macro = 0.842 (fit in 8.11s)

Experiment: Analyzer: word, ngram_range: (1, 1) - Features: 25,631
Logistic Regression: F1-macro = 0.842 (fit in 6.14s)

Experiment: Analyzer: word, ngram_range: (1, 2) - Features: 64,194
Logistic Regression: F1-macro = 0.846 (fit in 18.67s)

Experiment: Analyzer: word, ngram_range: (2, 2) - Features: 38,563
Logistic Regression: F1-macro = 0.713 (fit in 7.83s)


c:\Users\nallathambi\anaconda3\envs\tm\Lib\site-packages\sklearn\feature_extraction\text.py:539: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(



Experiment: Analyzer: char, ngram_range: (3, 5) - Features: 453,097
Logistic Regression: F1-macro = 0.850 (fit in 226.88s)

Experiment: Max features: 5000 - Features: 5,000
Logistic Regression: F1-macro = 0.803 (fit in 2.36s)

Experiment: Max features: 10000 - Features: 10,000
Logistic Regression: F1-macro = 0.825 (fit in 3.61s)

Experiment: Max features: 20000 - Features: 20,000
Logistic Regression: F1-macro = 0.839 (fit in 7.19s)

Experiment: Max features: All - Features: 25,631
Logistic Regression: F1-macro = 0.842 (fit in 4.40s)

Task 4: Parameter Experiments Table (on Logistic Regression + TF-IDF)


,Experiment,Precision,Recall,F1,Features
0,Lowercase: True,0.847,0.841,0.842,25631
1,Lowercase: False,0.846,0.839,0.840,30935
2,Stop words: With,0.847,0.841,0.842,25631
3,Stop words: Without,0.848,0.841,0.842,25914
4,"Analyzer: word, ngram_range: (1, 1)",0.847,0.841,0.842,25631
5,"Analyzer: word, ngram_range: (1, 2)",0.851,0.845,0.846,64194
6,"Analyzer: word, ngram_range: (2, 2)",0.724,0.709,0.713,38563
7,"Analyzer: char, ngram_range: (3, 5)",0.855,0.849,0.850,453097
8,Max features: 5000,0.807,0.802,0.803,5000
9,Max features: 10000,0.831,0.824,0.825,10000
